# 🏡 Architecture & Interior AI Studio — Google Colab Free GPU Server (T4 15GB)
### Hướng dẫn 1-Click:
1. Chọn menu **Runtime (Thời gian chạy)** > **Change runtime type (Thay đổi loại thời gian chạy)** > Chọn **T4 GPU** > Bấm **Save**.
2. Bấm nút **Play ▶ (Run cell)** bên dưới để tự động cài đặt và khởi động Server.
3. Khi chạy xong, dòng cuối cùng sẽ xuất hiện link: `🔗 YOUR PUBLIC WEB APP URL: https://xxxx.trycloudflare.com`.
4. Copy link đó dán vào phần **Cài Đặt > Colab GPU Server URL** trên Web App hoặc mở trực tiếp link đó trên trình duyệt!

In [ ]:
#@title 🚀 Bước 1: Tải & Cài Đặt ComfyUI + Models Kiến Trúc + Cloudflare Tunnel
import os, subprocess, time, urllib.request, json
from IPython.display import display, HTML, clear_output

print("⏳ [1/4] Đang khởi tạo môi trường GPU Google Colab...")
%cd /content
!apt-get update -qq && apt-get install -y -qq aria2

# 1. Clone ComfyUI Core nếu chưa có
if not os.path.exists("/content/ComfyUI"):
    print("⚡ Đang clone ComfyUI Core Engine...")
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

# 2. Clone Custom Nodes thiết yếu
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
if not os.path.exists("comfyui_controlnet_aux"):
    !git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git

# 3. Cài đặt Python Dependencies
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q xformers torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q requests pillow

# 4. Tải Cloudflared Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("🌐 Đang cài đặt Cloudflare Quick Tunnel...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 5. Tải Fast Checkpoints & ControlNet bằng Aria2 (Đa luồng tốc độ cao)
print("📥 [2/4] Đang tải AI Models kiến trúc (Realistic Vision & ControlNet Depth)...")
os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)
os.makedirs("/content/ComfyUI/models/controlnet", exist_ok=True)

# Realistic Vision V5.1
rv_path = "/content/ComfyUI/models/checkpoints/Realistic_Vision_V5.1.safetensors"
if not os.path.exists(rv_path):
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/checkpoints -o Realistic_Vision_V5.1.safetensors https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors

# ControlNet Depth SD1.5
cn_path = "/content/ComfyUI/models/controlnet/control_v11f1p_sd15_depth.pth"
if not os.path.exists(cn_path):
    !aria2c -x 16 -s 16 -k 1M -c -d /content/ComfyUI/models/controlnet -o control_v11f1p_sd15_depth.pth https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth

print("✅ Toàn bộ Models và Dependencies đã sẵn sàng!")


In [ ]:
#@title 🚀 Bước 2: Khởi Chạy ComfyUI + Backend API + Xuất URL Công Khai
import subprocess, time, re, os
from IPython.display import display, HTML

# 1. Khởi động ComfyUI chạy nền trên cổng 8188
print("⚡ [3/4] Đang khởi động ComfyUI Server trên GPU T4...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen(["python", "main.py", "--port", "8188", "--listen", "127.0.0.1", "--highvram", "--dont-print-server"])

time.sleep(5)

# 2. Khởi chạy Cloudflare Tunnel mở cổng 8188 công khai
print("🌐 [4/4] Đang tạo đường hầm bảo mật Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stderr=subprocess.PIPE, text=True)

public_url = None
for line in tunnel_proc.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    html_output = f"""
    <div style="background: linear-gradient(135deg, #0f172a, #1e293b); border: 2px solid #10b981; border-radius: 12px; padding: 20px; color: white; font-family: sans-serif; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
        <h2 style="color: #10b981; margin-top: 0;">🎉 COMYUI CLOUD GPU SERVER SẴN SÀNG!</h2>
        <p style="font-size: 15px;">Link kết nối GPU Cloud của bạn:</p>
        <div style="background: #020617; border: 1px solid #334155; padding: 12px; border-radius: 8px; font-family: monospace; font-size: 16px; color: #38bdf8; font-weight: bold; margin-bottom: 15px;">
            {public_url}
        </div>
        <p style="font-size: 13px; color: #94a3b8;">👉 Hãy copy đường link trên và dán vào ô <b>Cài Đặt (icon Bánh Răng) > Remote Server URL</b> trên Web App để bắt đầu render trực tiếp từ GPU Colab miễn phí!</p>
    </div>
    """
    display(HTML(html_output))
    print(f"\n🔗 YOUR PUBLIC COMFYUI SERVER URL: {public_url}\n")
else:
    print("❌ Không tìm thấy URL Cloudflare Tunnel, vui lòng kiểm tra lại log!")
